In [2]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../datasets/raw/queue/hourly.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (338555, 12)

Columns:
['airport', 'terminal', 'queue_type', 'hour_et', 'day_of_week', 'hour_of_day', 'avg_wait', 'p50_wait', 'p75_wait', 'p95_wait', 'max_wait', 'n_samples']


,airport,terminal,queue_type,hour_et,day_of_week,hour_of_day,avg_wait,p50_wait,p75_wait,p95_wait,max_wait,n_samples
0,BOS,Checkpoint 1: A Gates,precheck,2026-10-18 06:00:00,0,6,5.86,6,7,8,8,56
1,BOS,Checkpoint 1: A Gates,precheck,2026-10-18 07:00:00,0,7,3.31,4,4,6,6,55
2,BOS,Checkpoint 1: A Gates,standard,2026-08-14 09:00:00,5,9,7.67,7,8,9,9,6
3,BOS,Checkpoint 1: A Gates,standard,2026-08-14 10:00:00,5,10,8.13,8,9,14,16,64
4,BOS,Checkpoint 1: A Gates,standard,2026-08-14 11:00:00,5,11,9.47,9,11,14,16,59


In [2]:
pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 1.0 MB/s eta 0:00:0000:0100:010m
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../datasets/raw/queue/hourly.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (338555, 12)

Columns:
['airport', 'terminal', 'queue_type', 'hour_et', 'day_of_week', 'hour_of_day', 'avg_wait', 'p50_wait', 'p75_wait', 'p95_wait', 'max_wait', 'n_samples']


,airport,terminal,queue_type,hour_et,day_of_week,hour_of_day,avg_wait,p50_wait,p75_wait,p95_wait,max_wait,n_samples
0,BOS,Checkpoint 1: A Gates,precheck,2026-10-18 06:00:00,0,6,5.86,6,7,8,8,56
1,BOS,Checkpoint 1: A Gates,precheck,2026-10-18 07:00:00,0,7,3.31,4,4,6,6,55
2,BOS,Checkpoint 1: A Gates,standard,2026-08-14 09:00:00,5,9,7.67,7,8,9,9,6
3,BOS,Checkpoint 1: A Gates,standard,2026-08-14 10:00:00,5,10,8.13,8,9,14,16,64
4,BOS,Checkpoint 1: A Gates,standard,2026-08-14 11:00:00,5,11,9.47,9,11,14,16,59


In [4]:
import pandas as pd

# Ensure timestamps are parsed correctly
df["hour_et"] = pd.to_datetime(df["hour_et"], errors="coerce")

print("1. DATASET SIZE")
print("Rows, columns:", df.shape)

print("\n2. MISSING VALUES")
print(df.isna().sum().to_string())

print("\n3. TIME COVERAGE")
print("Start:", df["hour_et"].min())
print("End:  ", df["hour_et"].max())
print("Invalid timestamps:", df["hour_et"].isna().sum())

print("\n4. CHECKPOINT COVERAGE")
for col in ["airport", "terminal", "queue_type"]:
    print(f"{col}: {df[col].nunique()} unique values")

print("\n5. DUPLICATE CHECKPOINT-HOUR ROWS")
key = ["airport", "terminal", "queue_type", "hour_et"]
print("Duplicates:", df.duplicated(key).sum())

print("\n6. WAIT-TIME STATISTICS")
print(
    df[["avg_wait", "p50_wait", "p75_wait",
        "p95_wait", "max_wait", "n_samples"]]
    .describe()
    .round(2)
    .to_string()
)

print("\n7. INVALID VALUES")
print("Negative average waits:", (df["avg_wait"] < 0).sum())
print("Zero sample counts:", (df["n_samples"] == 0).sum())
print("Negative sample counts:", (df["n_samples"] < 0).sum())

print("\n8. CONSECUTIVE HOURLY OBSERVATIONS")
groups = ["airport", "terminal", "queue_type"]
sorted_df = df.sort_values(groups + ["hour_et"]).copy()
next_time = sorted_df.groupby(groups)["hour_et"].shift(-1)

valid_next_hour = (next_time - sorted_df["hour_et"]) == pd.Timedelta(hours=1)
print("Rows with an exact next-hour observation:", int(valid_next_hour.sum()))
print("Rows without an exact next-hour observation:", int((~valid_next_hour).sum()))

1. DATASET SIZE
Rows, columns: (338555, 12)

2. MISSING VALUES
airport        0
terminal       0
queue_type     0
hour_et        0
day_of_week    0
hour_of_day    0
avg_wait       0
p50_wait       0
p75_wait       0
p95_wait       0
max_wait       0
n_samples      0

3. TIME COVERAGE
Start: 2026-04-01 08:00:00
End:   2026-10-25 04:00:00
Invalid timestamps: 0

4. CHECKPOINT COVERAGE
airport: 30 unique values
terminal: 75 unique values
queue_type: 2 unique values

5. DUPLICATE CHECKPOINT-HOUR ROWS
Duplicates: 0

6. WAIT-TIME STATISTICS
        avg_wait   p50_wait   p75_wait   p95_wait   max_wait  n_samples
count  338555.00  338555.00  338555.00  338555.00  338555.00  338555.00
mean        5.74       5.55       6.64       8.21       8.74     105.21
std         6.39       6.45       7.34       8.64       9.00      37.66
min         0.00       0.00       0.00       0.00       0.00       1.00
25%         2.00       2.00       2.00       3.00       3.00      83.00
50%         3.85       4.00 

In [5]:
import numpy as np
import pandas as pd

# Work on a separate copy
queue_df = df.copy()

queue_df["hour_et"] = pd.to_datetime(
    queue_df["hour_et"], errors="raise"
)

CHECKPOINT = ["airport", "terminal", "queue_type"]

queue_df = (
    queue_df.sort_values(CHECKPOINT + ["hour_et"])
    .reset_index(drop=True)
)

# --------------------------------------------------
# 1. TARGET: average wait exactly one hour later
# --------------------------------------------------
groups = queue_df.groupby(CHECKPOINT, sort=False)

next_time = groups["hour_et"].shift(-1)
next_wait = groups["avg_wait"].shift(-1)

exact_next_hour = (
    next_time == queue_df["hour_et"] + pd.Timedelta(hours=1)
)

queue_df["target_wait_1h"] = next_wait.where(exact_next_hour)
queue_df["target_time"] = (
    queue_df["hour_et"] + pd.Timedelta(hours=1)
).where(exact_next_hour)

# --------------------------------------------------
# 2. LAG FEATURES: require exact time offsets
# --------------------------------------------------
groups = queue_df.groupby(CHECKPOINT, sort=False)

for lag in [1, 2, 3, 24, 168]:
    previous_time = groups["hour_et"].shift(lag)
    previous_wait = groups["avg_wait"].shift(lag)

    exact_lag = (
        queue_df["hour_et"] - previous_time
        == pd.Timedelta(hours=lag)
    )

    queue_df[f"avg_wait_lag_{lag}h"] = previous_wait.where(exact_lag)

# --------------------------------------------------
# 3. TREND AND SHORT-TERM FEATURES
# --------------------------------------------------
last_three = queue_df[
    ["avg_wait", "avg_wait_lag_1h", "avg_wait_lag_2h"]
]

# Only calculate the three-hour mean when all three
# consecutive hourly observations are available.
queue_df["avg_wait_mean_last3h"] = (
    last_three.mean(axis=1)
    .where(last_three.notna().all(axis=1))
)

queue_df["avg_wait_change_1h"] = (
    queue_df["avg_wait"] - queue_df["avg_wait_lag_1h"]
)

queue_df["avg_wait_change_3h"] = (
    queue_df["avg_wait"] - queue_df["avg_wait_lag_3h"]
)

# --------------------------------------------------
# 4. CYCLICAL FEATURES FOR THE FORECAST HOUR
# --------------------------------------------------
queue_df["target_hour"] = (
    queue_df["hour_of_day"] + 1
) % 24

# Dataset convention: Sunday=0, Monday=1, ..., Saturday=6
queue_df["target_day_of_week"] = (
    queue_df["day_of_week"]
    + (queue_df["hour_of_day"] == 23).astype(int)
) % 7

queue_df["target_hour_sin"] = np.sin(
    2 * np.pi * queue_df["target_hour"] / 24
)
queue_df["target_hour_cos"] = np.cos(
    2 * np.pi * queue_df["target_hour"] / 24
)

queue_df["target_day_sin"] = np.sin(
    2 * np.pi * queue_df["target_day_of_week"] / 7
)
queue_df["target_day_cos"] = np.cos(
    2 * np.pi * queue_df["target_day_of_week"] / 7
)

# --------------------------------------------------
# 5. KEEP ROWS WITH A VALID NEXT-HOUR TARGET
# --------------------------------------------------
queue_model_df = queue_df[
    queue_df["target_wait_1h"].notna()
].copy()

print("Original rows:", len(queue_df))
print("Rows with valid next-hour target:", len(queue_model_df))
print("Rows excluded because next hour is unavailable:",
      len(queue_df) - len(queue_model_df))

print("\nMissing values in engineered lag features:")
display(
    queue_model_df[
        [f"avg_wait_lag_{h}h" for h in [1, 2, 3, 24, 168]]
    ].isna().sum().to_frame("missing_count")
)

print("\nEngineered dataset preview:")
display(
    queue_model_df[
        CHECKPOINT
        + [
            "hour_et", "avg_wait", "avg_wait_lag_1h",
            "avg_wait_lag_2h", "avg_wait_mean_last3h",
            "avg_wait_change_1h", "target_time", "target_wait_1h"
        ]
    ].head(10)
)

Original rows: 338555
Rows with valid next-hour target: 331882
Rows excluded because next hour is unavailable: 6673

Missing values in engineered lag features:


,missing_count
avg_wait_lag_1h,6312
avg_wait_lag_2h,12443
avg_wait_lag_3h,18525
avg_wait_lag_24h,110076
avg_wait_lag_168h,158950



Engineered dataset preview:


,airport,terminal,queue_type,hour_et,avg_wait,avg_wait_lag_1h,avg_wait_lag_2h,avg_wait_mean_last3h,avg_wait_change_1h,target_time,target_wait_1h
0,BOS,Checkpoint 1: A Gates,precheck,2026-10-18 06:00:00,5.86,NaN,NaN,NaN,NaN,2026-10-18 07:00:00,3.31
2,BOS,Checkpoint 1: A Gates,standard,2026-08-14 09:00:00,7.67,NaN,NaN,NaN,NaN,2026-08-14 10:00:00,8.13
3,BOS,Checkpoint 1: A Gates,standard,2026-08-14 10:00:00,8.13,7.67,NaN,NaN,0.46,2026-08-14 11:00:00,9.47
4,BOS,Checkpoint 1: A Gates,standard,2026-08-14 11:00:00,9.47,8.13,7.67,8.423333,1.34,2026-08-14 12:00:00,9.08
5,BOS,Checkpoint 1: A Gates,standard,2026-08-14 12:00:00,9.08,9.47,8.13,8.893333,-0.39,2026-08-14 13:00:00,13.00
6,BOS,Checkpoint 1: A Gates,standard,2026-08-14 13:00:00,13.00,9.08,9.47,10.516667,3.92,2026-08-14 14:00:00,10.76
7,BOS,Checkpoint 1: A Gates,standard,2026-08-14 14:00:00,10.76,13.00,9.08,10.946667,-2.24,2026-08-14 15:00:00,11.67
8,BOS,Checkpoint 1: A Gates,standard,2026-08-14 15:00:00,11.67,10.76,13.00,11.810000,0.91,2026-08-14 16:00:00,5.95
9,BOS,Checkpoint 1: A Gates,standard,2026-08-14 16:00:00,5.95,11.67,10.76,9.460000,-5.72,2026-08-14 17:00:00,7.10
10,BOS,Checkpoint 1: A Gates,standard,2026-08-14 17:00:00,7.10,5.95,11.67,8.240000,1.15,2026-08-14 18:00:00,7.01


In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Use only rows with an exact next-hour target.
data = queue_model_df.copy()

# Split on unique timestamps so rows from the same hour
# cannot be scattered across different partitions.
unique_times = np.sort(data["hour_et"].unique())

train_cutoff = pd.Timestamp(unique_times[int(len(unique_times) * 0.70)])
val_cutoff = pd.Timestamp(unique_times[int(len(unique_times) * 0.85)])

# Keep each target inside its own partition's time window.
train = data[
    (data["hour_et"] < train_cutoff)
    & (data["target_time"] < train_cutoff)
].copy()

validation = data[
    (data["hour_et"] >= train_cutoff)
    & (data["hour_et"] < val_cutoff)
    & (data["target_time"] < val_cutoff)
].copy()

test = data[data["hour_et"] >= val_cutoff].copy()

print("TRAIN")
print(f"{len(train):,} rows | {train['hour_et'].min()} → {train['target_time'].max()}")

print("\nVALIDATION")
print(f"{len(validation):,} rows | {validation['hour_et'].min()} → {validation['target_time'].max()}")

print("\nTEST")
print(f"{len(test):,} rows | {test['hour_et'].min()} → {test['target_time'].max()}")

# Baseline: next-hour wait = current-hour wait.
y_val = validation["target_wait_1h"]
baseline_pred = validation["avg_wait"]

baseline_mae = mean_absolute_error(y_val, baseline_pred)
baseline_rmse = root_mean_squared_error(y_val, baseline_pred)

print("\nBASELINE — VALIDATION")
print(f"MAE:  {baseline_mae:.3f} minutes")
print(f"RMSE: {baseline_rmse:.3f} minutes")

TRAIN
152,181 rows | 2026-04-01 08:00:00 → 2026-08-26 21:00:00

VALIDATION
89,758 rows | 2026-08-26 22:00:00 → 2026-09-25 12:00:00

TEST
89,676 rows | 2026-09-25 13:00:00 → 2026-10-25 04:00:00

BASELINE — VALIDATION
MAE:  1.420 minutes
RMSE: 2.861 minutes


In [5]:
import time
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Identify input columns
categorical_features = [
    "airport",
    "terminal",
    "queue_type",
]

numeric_features = [
    "day_of_week",
    "hour_of_day",
    "avg_wait",
    "p50_wait",
    "p75_wait",
    "p95_wait",
    "max_wait",
    "n_samples",
    "avg_wait_lag_1h",
    "avg_wait_lag_2h",
    "avg_wait_lag_3h",
    "avg_wait_lag_24h",
    "avg_wait_lag_168h",
    "avg_wait_mean_last3h",
    "avg_wait_change_1h",
    "avg_wait_change_3h",
    "target_hour_sin",
    "target_hour_cos",
    "target_day_sin",
    "target_day_cos",
]

feature_columns = categorical_features + numeric_features
target_column = "target_wait_1h"

# Build train and validation matrices.
# The target timestamp and target value are never input features.
X_train = train[feature_columns]
y_train = train[target_column]

X_val = validation[feature_columns]
y_val = validation[target_column]

# Encode checkpoint identities; preserve numeric NaNs for HGB.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                dtype=np.float32,
            ),
            categorical_features,
        ),
        ("numeric", "passthrough", numeric_features),
    ],
    sparse_threshold=0,
)

queue_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=200,
                learning_rate=0.05,
                max_leaf_nodes=31,
                min_samples_leaf=30,
                l2_regularization=1.0,
                early_stopping=False,
                random_state=42,
            ),
        ),
    ]
)

# Train and time the model.
start = time.perf_counter()
queue_model.fit(X_train, y_train)
training_seconds = time.perf_counter() - start

# Predict on validation only, clipping impossible negative wait times.
val_predictions = np.clip(queue_model.predict(X_val), 0, None)

# Compare against the persistence baseline.
results = pd.DataFrame(
    [
        {
            "model": "Baseline: current wait",
            "MAE_minutes": mean_absolute_error(
                y_val, validation["avg_wait"]
            ),
            "RMSE_minutes": np.sqrt(
                mean_squared_error(y_val, validation["avg_wait"])
            ),
        },
        {
            "model": "HistGradientBoosting",
            "MAE_minutes": mean_absolute_error(
                y_val, val_predictions
            ),
            "RMSE_minutes": np.sqrt(
                mean_squared_error(y_val, val_predictions)
            ),
        },
    ]
)

print(f"Training time: {training_seconds:.2f} seconds")
display(results.round(3))

Training time: 23.42 seconds


,model,MAE_minutes,RMSE_minutes
0,Baseline: current wait,1.420,2.861
1,HistGradientBoosting,1.349,2.353


In [6]:
import time
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Preprocess categorical and numerical features separately.
rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "numeric",
            SimpleImputer(strategy="median", add_indicator=True),
            numeric_features,
        ),
    ]
)

rf_model = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=150,
                min_samples_leaf=5,
                max_features=0.8,
                random_state=42,
                n_jobs=2,
            ),
        ),
    ]
)

# Train on the training period only.
start = time.perf_counter()
rf_model.fit(X_train, y_train)
rf_training_seconds = time.perf_counter() - start

# Evaluate on validation data.
rf_predictions = np.clip(rf_model.predict(X_val), 0, None)

# Compare all three approaches.
comparison = pd.DataFrame([
    {
        "model": "Current-wait baseline",
        "MAE_minutes": mean_absolute_error(
            y_val, validation["avg_wait"]
        ),
        "RMSE_minutes": np.sqrt(
            mean_squared_error(y_val, validation["avg_wait"])
        ),
    },
    {
        "model": "HistGradientBoosting",
        "MAE_minutes": mean_absolute_error(
            y_val, val_predictions
        ),
        "RMSE_minutes": np.sqrt(
            mean_squared_error(y_val, val_predictions)
        ),
    },
    {
        "model": "Random Forest",
        "MAE_minutes": mean_absolute_error(
            y_val, rf_predictions
        ),
        "RMSE_minutes": np.sqrt(
            mean_squared_error(y_val, rf_predictions)
        ),
    },
])

print(f"Random Forest training time: {rf_training_seconds:.2f} seconds")
display(comparison.round(3))

KeyboardInterrupt: 

In [7]:
import time
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# -----------------------------------------
# 1. Select features
# -----------------------------------------
categorical_features = [
    "airport",
    "terminal",
    "queue_type",
]

numeric_features = [
    "day_of_week",
    "hour_of_day",
    "avg_wait",
    "p50_wait",
    "p75_wait",
    "p95_wait",
    "max_wait",
    "n_samples",
    "avg_wait_lag_1h",
    "avg_wait_lag_2h",
    "avg_wait_lag_3h",
    "avg_wait_lag_24h",
    "avg_wait_lag_168h",
    "avg_wait_mean_last3h",
    "avg_wait_change_1h",
    "avg_wait_change_3h",
    "target_hour_sin",
    "target_hour_cos",
    "target_day_sin",
    "target_day_cos",
]

feature_columns = categorical_features + numeric_features
target_column = "target_wait_1h"

X_train = train[feature_columns]
y_train = train[target_column]

X_val = validation[feature_columns]
y_val = validation[target_column]

# -----------------------------------------
# 2. Preprocessing
# -----------------------------------------
rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "numeric",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
            numeric_features,
        ),
    ],
    remainder="drop",
)

# -----------------------------------------
# 3. Smaller Random Forest
# -----------------------------------------
rf_model = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=40,
                max_depth=20,
                min_samples_leaf=10,
                max_features=0.8,
                max_samples=0.6,
                bootstrap=True,
                random_state=42,
                n_jobs=2,
            ),
        ),
    ]
)

# -----------------------------------------
# 4. Train
# -----------------------------------------
print("Training compact Random Forest...")
start_time = time.perf_counter()

rf_model.fit(X_train, y_train)

training_seconds = time.perf_counter() - start_time
print(f"Training completed in {training_seconds:.2f} seconds")

# -----------------------------------------
# 5. Validation predictions
# -----------------------------------------
start_time = time.perf_counter()

rf_predictions = np.clip(
    rf_model.predict(X_val),
    0,
    None,
)

prediction_seconds = time.perf_counter() - start_time

rf_mae = mean_absolute_error(y_val, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_predictions))

baseline_mae = mean_absolute_error(
    y_val,
    validation["avg_wait"],
)
baseline_rmse = np.sqrt(
    mean_squared_error(y_val, validation["avg_wait"])
)

# -----------------------------------------
# 6. Compare all models
# -----------------------------------------
results = [
    {
        "Model": "Current-wait baseline",
        "MAE (minutes)": baseline_mae,
        "RMSE (minutes)": baseline_rmse,
    },
    {
        "Model": "HistGradientBoosting",
        "MAE (minutes)": mean_absolute_error(
            y_val,
            val_predictions,
        ),
        "RMSE (minutes)": np.sqrt(
            mean_squared_error(y_val, val_predictions)
        ),
    },
    {
        "Model": "Compact Random Forest",
        "MAE (minutes)": rf_mae,
        "RMSE (minutes)": rf_rmse,
    },
]

results_df = pd.DataFrame(results)

print(f"\nPrediction time: {prediction_seconds:.2f} seconds")
display(results_df.round(3))

Training compact Random Forest...
Training completed in 186.05 seconds

Prediction time: 0.76 seconds


,Model,MAE (minutes),RMSE (minutes)
0,Current-wait baseline,1.420,2.861
1,HistGradientBoosting,1.349,2.353
2,Compact Random Forest,1.279,2.437


In [8]:
import tempfile
import time
from pathlib import Path

import joblib
import pandas as pd

models_to_compare = {
    "HistGradientBoosting": queue_model,
    "Random Forest": rf_model,
}

size_results = []

with tempfile.TemporaryDirectory() as temp_dir:
    for name, model in models_to_compare.items():
        model_path = Path(temp_dir) / f"{name.replace(' ', '_')}.joblib"

        start = time.perf_counter()
        joblib.dump(model, model_path, compress=3)
        save_seconds = time.perf_counter() - start

        size_mb = model_path.stat().st_size / (1024 ** 2)

        size_results.append({
            "Model": name,
            "Saved size (MB)": size_mb,
            "Save time (s)": save_seconds,
        })

size_df = pd.DataFrame(size_results)
display(size_df.round(3))

,Model,Saved size (MB),Save time (s)
0,HistGradientBoosting,0.299,0.136
1,Random Forest,6.793,0.575


In [9]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Combine training + validation data.
# Keep the test period completely separate.
train_val = pd.concat(
    [train, validation],
    ignore_index=True
).sort_values("hour_et")

X_train_val = train_val[feature_columns]
y_train_val = train_val["target_wait_1h"]

X_test = test[feature_columns]
y_test = test["target_wait_1h"]

# Retrain the selected compact model on all development data.
print("Training final HistGradientBoosting model...")

queue_model.fit(X_train_val, y_train_val)

print("Training complete.")

# Evaluate once on the held-out test period.
test_predictions = np.clip(
    queue_model.predict(X_test),
    0,
    None
)

baseline_predictions = test["avg_wait"]

results = pd.DataFrame([
    {
        "Model": "Current-wait baseline",
        "MAE_minutes": mean_absolute_error(
            y_test, baseline_predictions
        ),
        "RMSE_minutes": np.sqrt(
            mean_squared_error(y_test, baseline_predictions)
        ),
    },
    {
        "Model": "HistGradientBoosting",
        "MAE_minutes": mean_absolute_error(
            y_test, test_predictions
        ),
        "RMSE_minutes": np.sqrt(
            mean_squared_error(y_test, test_predictions)
        ),
    },
])

display(results.round(3))

baseline_mae = results.loc[0, "MAE_minutes"]
model_mae = results.loc[1, "MAE_minutes"]

print(
    f"Test MAE improvement over baseline: "
    f"{(1 - model_mae / baseline_mae) * 100:.2f}%"
)

Training final HistGradientBoosting model...
Training complete.


,Model,MAE_minutes,RMSE_minutes
0,Current-wait baseline,1.627,3.244
1,HistGradientBoosting,1.450,2.960


Test MAE improvement over baseline: 10.89%


In [10]:
from pathlib import Path
import json
import time
import joblib

# Save location
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "queue_wait_hgb_v1.joblib"
METADATA_PATH = MODELS_DIR / "queue_wait_hgb_v1_metadata.json"

# Save the complete pipeline, including preprocessing
start = time.perf_counter()
joblib.dump(queue_model, MODEL_PATH, compress=3)
save_seconds = time.perf_counter() - start

model_size_mb = MODEL_PATH.stat().st_size / (1024 ** 2)

# Final evaluation results
metadata = {
    "model_name": "Queue Waiting-Time HistGradientBoosting",
    "model_version": "queue-wait-hgb-v1",
    "model_type": "HistGradientBoostingRegressor",
    "model_scope": (
        "US airport checkpoint waiting-time prototype; "
        "not trained on Rahat Setu event data"
    ),
    "dataset": "US Airport Security & Immigration Wait Times",
    "prediction_target": "target_wait_1h",
    "target_description": (
        "Average waiting time at the same checkpoint exactly one hour later"
    ),
    "target_unit": "minutes",
    "forecast_horizon_hours": 1,
    "features": feature_columns,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "metrics": {
        "baseline_test_mae_minutes": float(baseline_mae),
        "model_test_mae_minutes": float(model_mae),
        "baseline_test_rmse_minutes": float(
            results.loc[0, "RMSE_minutes"]
        ),
        "model_test_rmse_minutes": float(
            results.loc[1, "RMSE_minutes"]
        ),
        "test_mae_improvement_pct": float(
            (1 - model_mae / baseline_mae) * 100
        ),
    },
    "training_rows": int(len(train_val)),
    "validation_rows": int(len(validation)),
    "test_rows": int(len(test)),
    "training_period_start": str(train_val["hour_et"].min()),
    "training_period_end": str(train_val["target_time"].max()),
    "test_period_start": str(test["hour_et"].min()),
    "test_period_end": str(test["target_time"].max()),
    "hyperparameters": {
        "max_iter": 200,
        "learning_rate": 0.05,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 30,
        "l2_regularization": 1.0,
        "early_stopping": False,
        "random_state": 42,
    },
    "saved_size_mb": round(model_size_mb, 4),
    "test_evaluation_note": (
        "Evaluated once on the held-out chronological test partition"
    ),
}

METADATA_PATH.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Model saved:", MODEL_PATH.resolve())
print("Metadata saved:", METADATA_PATH.resolve())
print(f"Artifact size: {model_size_mb:.3f} MB")
print(f"Save time: {save_seconds:.3f} seconds")

Model saved: /home/nayan/Consumer-Behavior-PBL/model-training/models/queue_wait_hgb_v1.joblib
Metadata saved: /home/nayan/Consumer-Behavior-PBL/model-training/models/queue_wait_hgb_v1_metadata.json
Artifact size: 0.301 MB
Save time: 0.220 seconds


In [11]:
import json
import joblib
import numpy as np
from pathlib import Path

MODEL_PATH = Path("../models/queue_wait_hgb_v1.joblib")
METADATA_PATH = Path("../models/queue_wait_hgb_v1_metadata.json")

# Reload the saved model and metadata
loaded_queue_model = joblib.load(MODEL_PATH)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    queue_metadata = json.load(f)

# Use actual held-out test rows for a smoke test
sample_X = X_test.iloc[:10]

original_predictions = queue_model.predict(sample_X)
reloaded_predictions = loaded_queue_model.predict(sample_X)

max_difference = np.max(
    np.abs(original_predictions - reloaded_predictions)
)

print("Model version:", queue_metadata["model_version"])
print("Features in metadata:", len(queue_metadata["features"]))
print("Artifact size (MB):", round(MODEL_PATH.stat().st_size / 1024**2, 3))
print("Maximum prediction difference:", max_difference)
print(
    "Predictions identical:",
    np.allclose(original_predictions, reloaded_predictions)
)

print("\nSample predictions (minutes):")
for actual, predicted in zip(
    y_test.iloc[:10].to_numpy(),
    reloaded_predictions
):
    print(f"Actual: {actual:7.2f} | Predicted: {max(0, predicted):7.2f}")

Model version: queue-wait-hgb-v1
Features in metadata: 23
Artifact size (MB): 0.301
Maximum prediction difference: 0.0
Predictions identical: True

Sample predictions (minutes):
Actual:    3.31 | Predicted:    6.17
Actual:    7.98 | Predicted:    7.99
Actual:    7.50 | Predicted:    7.90
Actual:    7.20 | Predicted:    7.34
Actual:    7.12 | Predicted:    7.20
Actual:    8.09 | Predicted:    7.10
Actual:    4.33 | Predicted:    6.62
Actual:    1.45 | Predicted:    3.84
Actual:    0.72 | Predicted:    1.47
Actual:    1.00 | Predicted:    0.82


In [9]:
import sys
import numpy as np

# Make model-training/src importable from the notebook
if "../src" not in sys.path:
    sys.path.insert(0, "../src")

from queue_predict import predict_queue_wait

# Take one real row from the held-out test features
sample_features = X_test.iloc[0].to_dict()

# Predict using the reusable inference module
result = predict_queue_wait(sample_features)

# Compare with the notebook's loaded pipeline
notebook_prediction = float(
    queue_model.predict(X_test.iloc[[0]])[0]
)

print("Inference module result:")
print(result)

print("\nNotebook prediction:", round(notebook_prediction, 4))
print(
    "Prediction matches:",
    np.isclose(
        result["predicted_wait_minutes"],
        max(0.0, notebook_prediction),
        atol=0.011,
    ),
)

NameError: name 'queue_model' is not defined

In [8]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd

# =========================================================
# 1. Locate the dataset, saved model, and metadata
# =========================================================
cwd = Path.cwd().resolve()

def find_file(candidates, description):
    for path in candidates:
        path = path.resolve()
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"Could not find {description}. Checked:\n"
        + "\n".join(str(p) for p in candidates)
    )

DATA_PATH = find_file([
    cwd / "../datasets/raw/queue/hourly.parquet",
    cwd / "datasets/raw/queue/hourly.parquet",
    cwd / "model-training/datasets/raw/queue/hourly.parquet",
], "hourly.parquet")

MODEL_PATH = find_file([
    cwd / "../models/queue_wait_hgb_v1.joblib",
    cwd / "models/queue_wait_hgb_v1.joblib",
    cwd / "model-training/models/queue_wait_hgb_v1.joblib",
], "saved queue model")

METADATA_PATH = find_file([
    cwd / "../models/queue_wait_hgb_v1_metadata.json",
    cwd / "models/queue_wait_hgb_v1_metadata.json",
    cwd / "model-training/models/queue_wait_hgb_v1_metadata.json",
], "queue model metadata")

print("Dataset:", DATA_PATH)
print("Model:", MODEL_PATH)

# Make queue_predict.py importable
SRC_DIR = find_file([
    cwd / "../src/queue_predict.py",
    cwd / "src/queue_predict.py",
    cwd / "model-training/src/queue_predict.py",
], "queue_predict.py").parent

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from queue_predict import predict_queue_wait

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

feature_columns = metadata["features"]
categorical_features = metadata["categorical_features"]
numeric_features = metadata["numeric_features"]

# =========================================================
# 2. Reload raw data and reconstruct features
# =========================================================
df = pd.read_parquet(DATA_PATH)
df["hour_et"] = pd.to_datetime(df["hour_et"], errors="raise")

CHECKPOINT = ["airport", "terminal", "queue_type"]

queue_df = (
    df.sort_values(CHECKPOINT + ["hour_et"])
      .reset_index(drop=True)
      .copy()
)

groups = queue_df.groupby(CHECKPOINT, sort=False)

# Target: waiting time exactly one hour later
next_time = groups["hour_et"].shift(-1)
next_wait = groups["avg_wait"].shift(-1)

exact_next_hour = (
    next_time == queue_df["hour_et"] + pd.Timedelta(hours=1)
)

queue_df["target_wait_1h"] = next_wait.where(exact_next_hour)
queue_df["target_time"] = (
    queue_df["hour_et"] + pd.Timedelta(hours=1)
).where(exact_next_hour)

# Exact-time historical lags
groups = queue_df.groupby(CHECKPOINT, sort=False)

for lag in [1, 2, 3, 24, 168]:
    previous_time = groups["hour_et"].shift(lag)
    previous_wait = groups["avg_wait"].shift(lag)

    exact_lag = (
        queue_df["hour_et"] - previous_time
        == pd.Timedelta(hours=lag)
    )

    queue_df[f"avg_wait_lag_{lag}h"] = previous_wait.where(exact_lag)

# Short-term averages and trends
last_three = queue_df[
    ["avg_wait", "avg_wait_lag_1h", "avg_wait_lag_2h"]
]

queue_df["avg_wait_mean_last3h"] = (
    last_three.mean(axis=1)
    .where(last_three.notna().all(axis=1))
)

queue_df["avg_wait_change_1h"] = (
    queue_df["avg_wait"] - queue_df["avg_wait_lag_1h"]
)

queue_df["avg_wait_change_3h"] = (
    queue_df["avg_wait"] - queue_df["avg_wait_lag_3h"]
)

# Cyclical features for the forecast hour and day
queue_df["target_hour"] = (
    queue_df["hour_of_day"] + 1
) % 24

queue_df["target_day_of_week"] = (
    queue_df["day_of_week"]
    + (queue_df["hour_of_day"] == 23).astype(int)
) % 7

queue_df["target_hour_sin"] = np.sin(
    2 * np.pi * queue_df["target_hour"] / 24
)
queue_df["target_hour_cos"] = np.cos(
    2 * np.pi * queue_df["target_hour"] / 24
)
queue_df["target_day_sin"] = np.sin(
    2 * np.pi * queue_df["target_day_of_week"] / 7
)
queue_df["target_day_cos"] = np.cos(
    2 * np.pi * queue_df["target_day_of_week"] / 7
)

# Keep only observations with a valid next-hour target
queue_model_df = queue_df[
    queue_df["target_wait_1h"].notna()
].copy()

missing_features = set(feature_columns) - set(queue_model_df.columns)
if missing_features:
    raise ValueError(f"Missing model features: {sorted(missing_features)}")

print("\nRows with valid next-hour target:", len(queue_model_df))

# =========================================================
# 3. Rebuild the chronological partitions
#    using the same 70% / 15% / 15% method
# =========================================================
unique_times = np.sort(queue_model_df["hour_et"].unique())

train_cutoff = pd.Timestamp(unique_times[int(len(unique_times) * 0.70)])
val_cutoff = pd.Timestamp(unique_times[int(len(unique_times) * 0.85)])

train = queue_model_df[
    (queue_model_df["hour_et"] < train_cutoff)
    & (queue_model_df["target_time"] < train_cutoff)
].copy()

validation = queue_model_df[
    (queue_model_df["hour_et"] >= train_cutoff)
    & (queue_model_df["hour_et"] < val_cutoff)
    & (queue_model_df["target_time"] < val_cutoff)
].copy()

test = queue_model_df[
    queue_model_df["hour_et"] >= val_cutoff
].copy()

X_test = test[feature_columns]
y_test = test["target_wait_1h"]

print(f"Training rows:   {len(train):,}")
print(f"Validation rows: {len(validation):,}")
print(f"Test rows:       {len(test):,}")
print("Test period:", test["hour_et"].min(), "to", test["target_time"].max())

# =========================================================
# 4. Load the saved model and compare predictions
# =========================================================
loaded_model = joblib.load(MODEL_PATH)

# First test row from the held-out test set
sample_X = X_test.iloc[[0]]
sample_features = sample_X.iloc[0].to_dict()

# Prediction from the saved pipeline
reference_prediction = float(loaded_model.predict(sample_X)[0])

# Prediction from the reusable inference module
result = predict_queue_wait(sample_features)

print("\n===== QUEUE MODEL SMOKE TEST =====")
print("Model version:", metadata["model_version"])
print("Artifact size (MB):", round(MODEL_PATH.stat().st_size / 1024**2, 3))
print("Actual next-hour wait:", round(float(y_test.iloc[0]), 2), "minutes")
print("Saved pipeline prediction:", round(max(0.0, reference_prediction), 4))
print("Inference module result:", result)

matches = np.isclose(
    result["predicted_wait_minutes"],
    max(0.0, reference_prediction),
    atol=0.011,
)

print("\nPrediction matches:", matches)

if not matches:
    raise AssertionError(
        "The inference function's prediction doesn't match the saved pipeline."
    )

print("\nSUCCESS: Saved model and reusable inference module agree.")

Dataset: /home/nayan/Consumer-Behavior-PBL/model-training/datasets/raw/queue/hourly.parquet
Model: /home/nayan/Consumer-Behavior-PBL/model-training/models/queue_wait_hgb_v1.joblib

Rows with valid next-hour target: 331882
Training rows:   152,181
Validation rows: 89,758
Test rows:       89,676
Test period: 2026-09-25 13:00:00 to 2026-10-25 04:00:00

===== QUEUE MODEL SMOKE TEST =====
Model version: queue-wait-hgb-v1
Artifact size (MB): 0.301
Actual next-hour wait: 3.31 minutes
Saved pipeline prediction: 6.1657
Inference module result: {'predicted_wait_minutes': 6.17, 'unit': 'minutes', 'horizon_hours': 1, 'model_version': 'queue-wait-hgb-v1', 'model_scope': 'US airport checkpoint waiting-time prototype; not trained on Rahat Setu event data'}

Prediction matches: True

SUCCESS: Saved model and reusable inference module agree.


In [10]:
import sys
import joblib
import numpy as np
from pathlib import Path

# Locate the saved model
MODEL_PATH = Path("../models/queue_wait_hgb_v1.joblib").resolve()

# Make queue_predict.py importable
SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from queue_predict import predict_queue_wait

# Load the saved pipeline as our reference
saved_model = joblib.load(MODEL_PATH)

# Select one held-out test example
sample_X = X_test.iloc[[0]]
sample_features = sample_X.iloc[0].to_dict()

# Predict using the reusable inference module
result = predict_queue_wait(sample_features)

# Predict using the directly loaded saved pipeline
reference_prediction = float(saved_model.predict(sample_X)[0])

print("Inference module result:")
print(result)

print("\nSaved pipeline prediction:", round(reference_prediction, 4))

matches = np.isclose(
    result["predicted_wait_minutes"],
    max(0.0, reference_prediction),
    atol=0.011,
)

print("Prediction matches:", matches)

if not matches:
    raise AssertionError("The predictions do not match!")

print("\nSUCCESS: Saved model and inference module agree.")

Inference module result:
{'predicted_wait_minutes': 6.17, 'unit': 'minutes', 'horizon_hours': 1, 'model_version': 'queue-wait-hgb-v1', 'model_scope': 'US airport checkpoint waiting-time prototype; not trained on Rahat Setu event data'}

Saved pipeline prediction: 6.1657
Prediction matches: True

SUCCESS: Saved model and inference module agree.


In [11]:
import sys
import numpy as np

# Make model-training/src importable from the notebook
if "../src" not in sys.path:
    sys.path.insert(0, "../src")

from queue_predict import predict_queue_wait

# Take one real row from the held-out test features
sample_features = X_test.iloc[0].to_dict()

# Predict using the reusable inference module
result = predict_queue_wait(sample_features)

# Compare with the notebook's loaded pipeline
notebook_prediction = float(
    queue_model.predict(X_test.iloc[[0]])[0]
)

print("Inference module result:")
print(result)

print("\nNotebook prediction:", round(notebook_prediction, 4))
print(
    "Prediction matches:",
    np.isclose(
        result["predicted_wait_minutes"],
        max(0.0, notebook_prediction),
        atol=0.011,
    ),
)

NameError: name 'queue_model' is not defined

In [12]:
from pathlib import Path
import joblib
import numpy as np
import sys

# Load the saved model directly
MODEL_PATH = Path("../models/queue_wait_hgb_v1.joblib").resolve()
saved_model = joblib.load(MODEL_PATH)

# Import the reusable prediction function
SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from queue_predict import predict_queue_wait

# Use the first held-out test example
sample_X = X_test.iloc[[0]]
sample_features = sample_X.iloc[0].to_dict()

# Predict through the inference module
result = predict_queue_wait(sample_features)

# Predict directly through the saved model
direct_prediction = float(saved_model.predict(sample_X)[0])

print("Inference module result:")
print(result)

print("\nDirect saved-model prediction:", round(direct_prediction, 4))

matches = np.isclose(
    result["predicted_wait_minutes"],
    max(0.0, direct_prediction),
    atol=0.011,
)

print("Prediction matches:", matches)

assert matches, "Predictions do not match!"
print("\nSUCCESS: Both prediction paths agree.")

Inference module result:
{'predicted_wait_minutes': 6.17, 'unit': 'minutes', 'horizon_hours': 1, 'model_version': 'queue-wait-hgb-v1', 'model_scope': 'US airport checkpoint waiting-time prototype; not trained on Rahat Setu event data'}

Direct saved-model prediction: 6.1657
Prediction matches: True

SUCCESS: Both prediction paths agree.
